In [6]:
# Import librerie e setup percorsi
from datetime import datetime
import glob
import json
import os
import sys
import time
import uuid

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from langchain_community.embeddings import FastEmbedEmbeddings

# Import del modulo custom src/
PROJECT_ROOT = os.path.expanduser("~/tesi_graphrag")
sys.path.append(PROJECT_ROOT)
from src.config import (
    EMBEDDING_MODEL,
    QDRANT_URL,
    LLM_MODEL,
    OLLAMA_URL,
)

# Configurazione ingestione e salvataggio registro
REGISTRY_PATH = os.path.join(PROJECT_ROOT, "data/collection_registry.json")
CHUNK_SIZE = 450
CHUNK_OVERLAP = 90

In [7]:
# Percorsi e configurazioni -- DataSet Tesi (dst) -- Cambiare se necessario
DATASET_DIR = os.path.join(PROJECT_ROOT, "data/raw/dst")
COLLECTION_NAME = "ds_tesi_multilang"

print(f"Dataset directory: {DATASET_DIR} | Embeggind model: {EMBEDDING_MODEL}")

Dataset directory: /home/jovyan/tesi_graphrag/data/raw/dst | Embeggind model: sentence-transformers/paraphrase-multilingual-mpnet-base-v2


In [8]:
#! trova e elenca tutti i file contenuti nel percorso specificato
pdf_files = glob.glob(os.path.join(DATASET_DIR, "*.pdf"))
print(f"Trovati {len(pdf_files)} PDF in {DATASET_DIR}")

if not pdf_files:
    sys.exit(f"Nessun PDF trovato in {DATASET_DIR}. Controlla il percorso prima di procedere.")

# Configurazione Text Splitter (256 token / 1000ca. caratteri, overlap di 150 - misurato in caratteri) 
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, length_function=len
)

all_docs = []
for pdf_path in pdf_files:
    filename = os.path.basename(pdf_path).replace(".pdf", "").strip()

    try:
        loader = PyMuPDFLoader(pdf_path)
        pages = loader.load()
    except Exception as e:
        print(f"!!!>  Errore nel caricamento di '{filename}': {e} >>> file saltato")
        continue
    
    # Split in chunk
    chunks = text_splitter.split_documents(pages)

    for idx, chunk in enumerate(chunks):
        page_num = chunk.metadata.get("page", 0)

        chunk_id = f"{filename}_chunk_{idx}"

        # Mantiene i metadati originali e aggiunge i nuovi campi di interesse
        chunk.metadata.update(
            {
                "chunk_id": chunk_id,
                "doc_id": filename,
                "page": int(page_num),
                "chunk_index": int(idx),
                "user_tags": [],
            }
        )

        all_docs.append(chunk)

print(f"Totale chunk generati: {len(all_docs)}")

Trovati 6 PDF in /home/jovyan/tesi_graphrag/data/raw/dst
Totale chunk generati: 1276


In [9]:
# Indicizzazione vettoriale su Qdrant
client = QdrantClient(url=QDRANT_URL)
embeddings = FastEmbedEmbeddings(model_name=EMBEDDING_MODEL)

# Calcola automaticamente la dimensione corretta del modello attivo (es. 768)
vector_dim = len(embeddings.embed_query("test"))

# Crea la collezione solo se non esiste già
#  Se già esistente, gli uuid deterministici calcolati la modificheranno con l'upsert
if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=vector_dim, distance=Distance.COSINE),
    )


vectorstore = QdrantVectorStore(
    client=client, collection_name=COLLECTION_NAME, embedding=embeddings
)

# Caricamento di tutti i chunk su Qdrant
#  Inserimento in batch di 25 chunk alla volta per vedere
#  se il processo si è bloccato o sta funzionando; + timer
batch_size = 128
total_docs = len(all_docs)
total_start_time = time.perf_counter()

print(
    f"\n>> Avvio vettorizzazione di {total_docs} chunk ({batch_size} chunk per batch)"
)

# Ciclo di inserimento in batch con timer parziali e progressivo
for batch_idx, i in enumerate(range(0, total_docs, batch_size), start=1):
    batch_start_time = time.perf_counter()

    batch = all_docs[i : i + batch_size]

    # ID deterministico basato su namespace UUID + chunk_id
    #  Permette di riaggiornare la collection tramite upsert
    batch_ids = [
        str(uuid.uuid5(uuid.NAMESPACE_DNS, doc.metadata["chunk_id"]))
        for doc in batch
    ]

    # Inserimento se assetne o upsert implicito se giù esistente il corrispettivo batch_id
    #  NB: se dovessero essere cambiate le dimensioni dei chunk, l'upsert non cancellerà i vecchi punti da Qdrant
    vectorstore.add_documents(documents=batch, ids=batch_ids)

    batch_elapsed = time.perf_counter() - batch_start_time
    processed_count = min(i + batch_size, total_docs)

    print(
        f"   [Batch {batch_idx:02d}] Indicizzati {processed_count:3d}/{total_docs} chunk "
        f"<T> tempo batch: {batch_elapsed:.2f} s"
    )

# Tempo totale
total_elapsed = time.perf_counter() - total_start_time

# Salvataggio associazione collection - embedding_model
#  Ogni collection deve infatti essere manipolata dallo stesso embedding_model
#  che ne ha fatto l'Ingestion
registry_data = {}
if os.path.exists(REGISTRY_PATH):
    try:
        with open(REGISTRY_PATH, "r", encoding="utf-8") as f:
            registry_data = json.load(f)
    except Exception:
        registry_data = {}

registry_data[COLLECTION_NAME] = {
    "embedding_model": EMBEDDING_MODEL,
    "vector_dimension": vector_dim,
    "distance_metric": "COSINE",
    "total_chunks": total_docs,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    #"hypothetical_tags": sorted(list(dataset_global_tags)),
    "updated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
}

os.makedirs(os.path.dirname(REGISTRY_PATH), exist_ok=True)
with open(REGISTRY_PATH, "w", encoding="utf-8") as f:
    json.dump(registry_data, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 55)
print(
    f"!!!> COMPLETATO! Inseriti {total_docs} chunk nella collezione '{COLLECTION_NAME}'"
)
print(f"<TT> TEMPO TOTALE VETTORIZZAZIONE & UPLOAD: {total_elapsed:.2f} s")
print("=" * 55)

/home/jovyan/tesi_graphrag/.venv/lib/python3.12/site-packages/langchain_community/embeddings/fastembed.py:109: UserWarning: The model sentence-transformers/paraphrase-multilingual-mpnet-base-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  values["model"] = fastembed.TextEmbedding(



>> Avvio vettorizzazione di 1276 chunk (128 chunk per batch)
   [Batch 01] Indicizzati 128/1276 chunk <T> tempo batch: 29.50 s
   [Batch 02] Indicizzati 256/1276 chunk <T> tempo batch: 25.44 s
   [Batch 03] Indicizzati 384/1276 chunk <T> tempo batch: 33.84 s
   [Batch 04] Indicizzati 512/1276 chunk <T> tempo batch: 25.64 s
   [Batch 05] Indicizzati 640/1276 chunk <T> tempo batch: 25.32 s
   [Batch 06] Indicizzati 768/1276 chunk <T> tempo batch: 24.00 s
   [Batch 07] Indicizzati 896/1276 chunk <T> tempo batch: 41.12 s
   [Batch 08] Indicizzati 1024/1276 chunk <T> tempo batch: 40.71 s
   [Batch 09] Indicizzati 1152/1276 chunk <T> tempo batch: 40.59 s
   [Batch 10] Indicizzati 1276/1276 chunk <T> tempo batch: 23.13 s

!!!> COMPLETATO! Inseriti 1276 chunk nella collezione 'ds_tesi_multilang'
<TT> TEMPO TOTALE VETTORIZZAZIONE & UPLOAD: 309.30 s
